# 2-5 Excel への書き出し

ここまでで読み込み・集計・絞り込み・加工を一通りできるようになりました。仕上げは **書き出し** です。

DataFrame を Excel ファイル (`.xlsx`) に保存します。**複数のシート** にまとめて 1 つのレポートにする方法も扱います。これは第 4 章の総集編で書く **月次売上レポート** にそのままつながります。

## このノートのゴール

- `df.to_excel()` で **1 シートに保存** できる
- `index=False` の意味と使いどころが分かる
- `pd.ExcelWriter` で **複数シートを 1 ファイルにまとめる**
- `to_csv` での **文字化け対策**（`utf-8-sig`）が分かる
- 保存先のフォルダを **`os.makedirs`** で準備できる

## Excel / VBA との対応表

| やりたいこと | Excel / VBA | pandas |
|---|---|---|
| 1 シートに保存 | `Workbooks.Add` + `SaveAs` | `df.to_excel("path.xlsx", index=False)` |
| シート名を指定 | `Sheets("Sheet1").Name = "集計"` | `to_excel(..., sheet_name="集計")` |
| 複数シートに分ける | 複数の `Worksheets.Add` | `ExcelWriter` で `with` ブロック |
| CSV で保存 | 名前を付けて保存 → CSV | `df.to_csv("path.csv", index=False)` |
| フォルダを用意 | `MkDir` | `os.makedirs(path, exist_ok=True)` |

## データの準備

2-3、2-4 と同じ売上データを読み込みます。
また、**出力先のフォルダ** (`output/`) を準備します。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
BASE = "/content/drive/MyDrive/python-data-basics"
DATA = f"{BASE}/data/pandas"
OUT  = f"{BASE}/output"

# 出力フォルダが無ければ作る（既にあれば何もしない）
os.makedirs(OUT, exist_ok=True)

df = pd.read_excel(f"{DATA}/sales_2026-01.xlsx", sheet_name="売上", parse_dates=["date"])
print(f"{len(df)} 件読み込み、出力先: {OUT}")
df.head()

## 1. 最小限の書き出し

1 行だけで OK です。

In [ ]:
df.to_excel(f"{OUT}/sales_output.xlsx", index=False)
print("✅ 保存しました")
print("→ マイドライブ → python-data-basics → output → sales_output.xlsx を開いて確認")

### `index=False` の意味

pandas の DataFrame には、左端に **行番号（index）** が付いています。

- `index=False` 付き → 行番号は **書き出さない**（普通はこっち）
- `index=False` 無し → 左端に「Unnamed: 0」みたいな **不要な列が増えて** しまう

**集計表のように行名（商品名など）がインデックスになっているときは、逆に `index=False` を付けない** ことに注意。

In [ ]:
# (A) シート名を指定して保存
df.to_excel(f"{OUT}/sales_named.xlsx", sheet_name="2026年1月", index=False)

# (B) groupby の結果は、商品名が「行ラベル (index)」になっている
#     → このときは index=False を付けない方が正しい
by_product = df.groupby("product_name")["amount"].agg(["count", "sum", "mean"]).round(0)
print("商品名はインデックスとして残っている:")
print(by_product)

by_product.to_excel(f"{OUT}/by_product.xlsx")  # index=False を付けない
print("\n✅ 商品別集計を保存")

## 補足: 並べ替え (`sort_values`)

集計結果を **「合計が多い順」「単価が安い順」** に並べ替えてから保存したい、ということはよくあります。

書き方:
```python
df.sort_values("並べ替えに使う列名", ascending=False)
```

- `ascending=False` で **降順**（多い → 少ない）
- 省略すると **昇順**（少ない → 多い）

Excel の「データ → 並べ替え」と同じです。

In [ ]:
# 商品別集計を、売上合計の多い順に並べ替え
by_product.sort_values("sum", ascending=False)

## 2. 複数シートを 1 ファイルにまとめる (`ExcelWriter`)

実務では、**「1 つの Excel ファイルに、明細・集計・グラフ用データを分けて入れたい」** ことがよくあります。

そのときは `pd.ExcelWriter` を **`with` ブロック** で使います。

```python
with pd.ExcelWriter("path.xlsx", engine="openpyxl") as writer:
    df1.to_excel(writer, sheet_name="...", index=False)
    df2.to_excel(writer, sheet_name="...")
```

`with` ブロックを抜けるタイミングで、ファイルが **自動的に保存・クローズ** されます。

In [ ]:
# 3 種類のデータを準備
by_product = df.groupby("product_name")["amount"].agg(["count", "sum", "mean"]).round(0)
big_orders = df[df["amount"] >= 10000]

with pd.ExcelWriter(f"{OUT}/report_2026-01.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="明細", index=False)
    by_product.to_excel(writer, sheet_name="商品別集計")        # index 残す
    big_orders.to_excel(writer, sheet_name="大口取引", index=False)

print("✅ 3 シートのレポートを保存しました: report_2026-01.xlsx")

## 3. CSV にも書き出せる (`to_csv`)

**他のシステムに渡す** ときは CSV がよく使われます。

ここで注意なのが **文字コード**:

| 用途 | 指定 |
|---|---|
| Excel で開くつもり | `encoding="utf-8-sig"` ← **BOM 付き UTF-8** にしないと文字化け |
| プログラム同士の受け渡し | `encoding="utf-8"` |
| 古い Windows 業務システムに渡す | `encoding="cp932"` (≒ Shift_JIS) |

2-2 では **読み込み** 側で `cp932` を扱いました。書き出すときも同じ知識が活きます。

In [ ]:
# Excel で開く前提なら utf-8-sig が安全
df.to_csv(f"{OUT}/sales_output.csv", index=False, encoding="utf-8-sig")
print("✅ CSV を保存（Excel で開いて文字化けしないか確認してみてください）")

## 練習問題

1. `by_product` を **売上合計 (`sum`) が多い順** に並べてから保存してください（ヒント: `.sort_values("sum", ascending=False)`）
2. **日付ごとの売上合計** を求めて、`daily_sales.xlsx` という名前で保存してください
3. **`report_by_rank.xlsx`** という 1 ファイルに、`amount >= 20000`（大口）、`10000 <= amount < 20000`（中口）、`amount < 10000`（小口）の 3 つを **別シート** で保存してください

In [ ]:
# ここに書いてみてください



<details>
<summary>答えを見る</summary>

```python
# 1. 売上合計の多い順に並べて保存
by_product.sort_values("sum", ascending=False).to_excel(f"{OUT}/by_product_sorted.xlsx")

# 2. 日付ごとの合計
daily = df.groupby("date")["amount"].sum()
daily.to_excel(f"{OUT}/daily_sales.xlsx")

# 3. ランク別に 3 シート
big    = df[df["amount"] >= 20000]
middle = df[(df["amount"] >= 10000) & (df["amount"] < 20000)]
small  = df[df["amount"] < 10000]

with pd.ExcelWriter(f"{OUT}/report_by_rank.xlsx", engine="openpyxl") as writer:
    big.to_excel(writer,    sheet_name="大口", index=False)
    middle.to_excel(writer, sheet_name="中口", index=False)
    small.to_excel(writer,  sheet_name="小口", index=False)
```

</details>

## よくあるエラー

### 1. `PermissionError: [Errno 13] Permission denied: '...xlsx'`
→ **同じファイルを Excel で開いたまま** 書き出そうとした。Excel を閉じて再実行。

### 2. `FileNotFoundError: ...output/...`
→ **保存先のフォルダがまだ無い**。`os.makedirs(OUT, exist_ok=True)` を先に走らせる。

### 3. シート名が長すぎる
→ Excel のシート名は **31 文字以内**。日本語でも 31 文字を超えるとエラーになる。

### 4. CSV を Excel で開いたら日本語が文字化け
→ `encoding="utf-8-sig"` を指定する（BOM 付き UTF-8）。

### 5. `ExcelWriter` を `with` 無しで使うと中身が空に見える
```python
writer = pd.ExcelWriter("x.xlsx")
df.to_excel(writer, sheet_name="A")
# writer.close() を忘れるとファイルが完成しない
```
→ **必ず `with pd.ExcelWriter(...) as writer:` の形** で使う。

## 第 2 章まとめ

これで pandas の基本一周が終わりました。

- **2-1** DataFrame の構造（行・列・index）
- **2-2** Excel / CSV の読み込み（`parse_dates`、文字コード、シート指定）
- **2-3** 集計と絞り込み（`groupby` / boolean indexing）
- **2-4** データの加工（列追加、`.dt` / `.str`、欠損値、`apply`）
- **2-5** Excel への書き出し（`to_excel`、マルチシート、文字コード）

**「Excel を読んで → 集計・加工して → Excel に書き出す」** の一周は、これで全部できます。

次は **第 3 章「matplotlib で可視化」** に進み、最後に **第 4 章「月次売上レポート自動生成」** で全部つなぎます。